In [10]:
import numpy as np
import pandas as pd

In [12]:
df = pd.read_csv('spam.csv', encoding='cp1252')

In [14]:
df.sample(5)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
4339,ham,I want to sent &lt;#&gt; mesages today. Thats...,NaN,NaN,NaN
6,ham,Even my brother is not like to speak with me. ...,NaN,NaN,NaN
4642,ham,Sorry i din lock my keypad.,NaN,NaN,NaN
4647,ham,We are okay. Going to sleep now. Later,NaN,NaN,NaN
1822,ham,If you're thinking of lifting me one then no.,NaN,NaN,NaN


In [16]:
df.shape

(5572, 5)

## 1. Data Cleaning
## 2. EDA
## 3. Text Preprocessing
## 4. Model Building
## 5. Evaluation

### 1. Data Cleaning

In [209]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 5169 entries, 0 to 5571
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   target          5169 non-null   int32 
 1   text            5169 non-null   object
 2   num_characters  5169 non-null   int64 
 3   Num_Words       5169 non-null   int64 
 4   num_sentences   5169 non-null   int64 
 5   num_words       5169 non-null   int64 
 6   new text        5169 non-null   object
dtypes: int32(1), int64(4), object(2)
memory usage: 302.9+ KB


In [211]:
# drop last three coloums
df.drop(columns = ['Unnamed: 2' , 'Unnamed: 3', 'Unnamed: 4'], inplace = True)

KeyError: "['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'] not found in axis"

In [ ]:
df.sample(5)

In [ ]:
#renaming the coloumns

df.rename(columns = {'v1': 'target', 'v2': 'text'}, inplace= True)
df.sample(10)

In [ ]:
from sklearn.preprocessing import LabelEncoder
encoder= LabelEncoder()

In [ ]:
df['target']= encoder.fit_transform(df['target'])

In [ ]:
df.head()

In [ ]:
#missing values
df.isnull().sum()

In [ ]:
#check for duplicate values
df.duplicated().sum()

In [ ]:
#remove duplicates
df = df.drop_duplicates(keep= 'first')

In [ ]:
df.duplicated().sum()

In [ ]:
df.shape

## 2.EDA

In [ ]:
df.head()

In [ ]:
df['target'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
plt.pie(df['target'].value_counts(), labels= ['ham', 'spam'], autopct = "%0.2f")
plt.show()

In [ ]:
#Data is imbalanced

In [ ]:
import nltk

In [ ]:
nltk.download('punkt')

In [ ]:
df['num_characters'] = df['text'].apply(len)

In [ ]:
df.head()

In [ ]:
import nltk
nltk.download(['punkt', 'punkt_tab'])

In [ ]:
import nltk

# Specify a directory for the punkt tokenizer if needed
nltk.data.path.append("C:/Users/pande/nltk_data")

# Now tokenize
df['Num_Words'] = df['text'].apply(lambda x: len(nltk.word_tokenize(x)))



In [ ]:
df.head()

In [ ]:
df['num_sentences'] = df['text'].apply(lambda x: len(nltk.sent_tokenize(x)))

In [ ]:
df.head()

In [ ]:
df['num_characters'] = df['text'].apply(len)
df['num_words'] = df['text'].apply(lambda x: len(x.split()))
df['num_sentences'] = df['text'].apply(lambda x: len(nltk.sent_tokenize(x)))


In [ ]:
#ham
df[df['target'] == 0][['num_characters', 'Num_Words', 'num_sentences']].describe()

In [ ]:
#spam
df[df['target'] == 1][['num_characters', 'Num_Words', 'num_sentences']].describe()


In [ ]:
import seaborn as sns

In [ ]:
df.columns
df.head()


In [ ]:
plt.figure(figsize = (12, 6))
sns.histplot(df[df['target'] == 0]['num_characters'], color = 'green')
sns.histplot(df[df['target'] == 1]['num_characters'], color = 'red')

In [ ]:
sns.pairplot(df, hue="target")


In [ ]:
df_numeric = df[['Num_Words', 'num_sentences', 'num_characters', 'target']]
correlation_matrix = df_numeric.corr()
print(correlation_matrix)


In [ ]:
df_numeric = df[[ 'target', 'num_characters',  'num_sentences', 'Num_Words']]
correlation_matrix = df_numeric.corr()
sns.heatmap(correlation_matrix, annot=True)


## 3. Data Preprocessing 
###  a. Lower case 
###  b. Tokenization
###  c. Removing special characters 
###  d. Removing stop words and punctuation 
###  e. Stemming

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

import nltk

ps = PorterStemmer()

def transform_text(text):
    text = text.lower()  #lower case conversion
    text = nltk.word_tokenize(text)  #tokenisation
    ans = []
    # Remove Non-Alphanumeric Token
    
    for ele in text:    
        if ele.isalnum():
            ans.append(ele)
    text = ans.copy()
    
     #Remove Stopwords
    ans.clear()
    for ele in text:
        if ele not in stopwords.words('english'):
            ans.append(ele)
    text = ans.copy()
    
    #Perform Stemming
    ans.clear()
    for ele in text:
        ele = ps.stem(ele)
        ans.append(ele)
    return ' '.join(ans)


nltk.download('stopwords')


In [ ]:
transform_text("I'm gonna be home soon and i don't want to talk about this stuff anymore tonight, k? I've cried enough today")

In [ ]:
from nltk.stem.porter import PorterStemmer
ps= PorterStemmer()

In [ ]:
df['new text']= df['text'].apply(transform_text)


In [ ]:
df.head()

In [ ]:
pip install wordcloud


In [ ]:
#word cloud to count which words are appearing more in spam or ham
from wordcloud import WordCloud
wc= WordCloud(width=1000, height=1000, min_font_size=10, background_color="white")

In [ ]:
spam= df[df['target']==1]['new text']
spam_str=' '.join(spam)
spam_wc= wc.generate(spam_str)

In [ ]:
plt.figure(figsize= (12,6))
plt.imshow(spam_wc)

In [ ]:
# For Ham Messages

ham= df[df['target']==0]['new text']
ham_str=' '.join(ham)
ham_wc= wc.generate(ham_str)

In [ ]:
plt.figure(figsize= (12,6))
plt.imshow(ham_wc)

In [ ]:
df.head()

In [ ]:
#now our goal to find 30 top spam words and ham word

spam_corpus = []
for msg in df[df['target'] == 1]['new text'].tolist():
    for words in msg.split():
        spam_corpus.append(words)

In [ ]:
len(spam_corpus)

In [ ]:
ham_list= ham.tolist()
ham_corpus= []
for item in ham_list:
    for word in item.split():
        ham_corpus.append(word)

len(ham_corpus)

In [ ]:
from collections import Counter
data = pd.DataFrame(Counter(spam_corpus).most_common(30), columns=['Word', 'Count'])

sns.barplot(x=data['Word'], y=data['Count'])
plt.xticks(rotation='vertical')
plt.title("Top 30 Words in Spam Messages")
plt.show()

In [ ]:
data = pd.DataFrame(Counter(ham_corpus).most_common(30), columns=['Word', 'Count'])

sns.barplot(x=data['Word'], y=data['Count'])
plt.xticks(rotation='vertical')
plt.title("Top 30 Words in Ham Messages")
plt.show()


In [ ]:
data = pd.DataFrame(Counter(ham_corpus).most_common(30), columns=['Word', 'Count'])

sns.barplot(x=data['Word'], y=data['Count'])
plt.xticks(rotation='vertical')
plt.show()


## 4 MODEL BUILDING

In [ ]:
#transformed text conversion to number -> vectors
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
cv= CountVectorizer()
tfidf= TfidfVectorizer(max_features=3000)

In [ ]:
#sparse array to dense array

X= tfidf.fit_transform(df['new text']).toarray()
X.shape

In [ ]:
y= df['target'].values
y.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score

In [ ]:
gnb= GaussianNB()
mnb= MultinomialNB()
bnb= BernoulliNB()

In [ ]:
gnb.fit(X_train, y_train)
y_pred1= gnb.predict(X_test)
print(accuracy_score(y_test, y_pred1))
print(confusion_matrix(y_test, y_pred1))
print(precision_score(y_test, y_pred1))

In [ ]:
mnb.fit(X_train, y_train)
y_pred2= mnb.predict(X_test)
print(accuracy_score(y_test, y_pred2))
print(confusion_matrix(y_test, y_pred2))
print(precision_score(y_test, y_pred2))

In [198]:
bnb.fit(X_train, y_train)
y_pred3 = bnb.predict(X_test)
print(accuracy_score(y_test, y_pred3))
print(confusion_matrix(y_test, y_pred3))
print(precision_score(y_test, y_pred3))

0.9835589941972921
[[895   1]
 [ 16 122]]
0.991869918699187


In [199]:
while True:
    email = input("\nEnter a email (or type 'exit' to quit): ").strip()
    
    if email.lower() == 'exit':
        print("Exiting the program. Goodbye!")
        break
    
    else:
        
        processed_email = transform_text(email)
      
        email_tfidf = tfidf.transform([processed_email])
      
        prediction = mnb.predict(email_tfidf)
    
        print("The email is classified as:", "SPAM (1)" if prediction[0] == 1 else "NOT SPAM (0)")


Enter a email (or type 'exit' to quit):  exit


Exiting the program. Goodbye!
